<a href="https://colab.research.google.com/github/Sasindu99-ai/Statistical-Learning-e22445/blob/main/Assignments/Assignment%207d/e22445_Bayesian_Inference_Assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<pre style="color:green;">
Assignment 7    :   Bayesian Inference 7d
Course          :   ME2050
Reg. No.        :   E/22/445
Name            :   P.M.S.S. Wjethunga
</pre>

---

# Q Bayesian Estimations for Structural Health Monitoring via Bounded Grid Updates

In aerospace and civil engineering, Structural Health Monitoring (SHM) is critical for detecting damage before a catastrophic failure occurs. Consider an aircraft wing or a bridge girder equipped with specialized vibration sensors. Over time, environmental fatigue or dynamic impacts can cause micro-fractures, resulting in a reduction of the component's mechanical stiffness.

Let $\Theta = \theta$ represent the structural **remaining stiffness efficiency factor**, where $\theta$ is physically bounded to the interval:

$$\theta \in (0, 1]$$

* $\theta = 1.0$ indicates a perfectly pristine, undamaged structural component.
* $\theta \to 0$ signifies critical degradation or severe structural cracking.

Let $K_{\text{nominal}}$ be the known, baseline stiffness of the structural component when it is entirely healthy. At each sequential inspection time step $k$ (where $k = 1, 2, \dots, n$), a sensor collects a noisy experimental stiffness measurement $y_k$.

Engineers model the degradation physics via a non-linear relationship with multiplicative log-normal measurement noise to prevent non-physical negative values:

$$y_k = \theta \cdot K_{\text{nominal}} \cdot e^{\epsilon_k}, \qquad \epsilon_k \sim \mathscr{N}(0, \sigma^2)$$

where $\sigma$ is the standard deviation of the sensor noise in log-space.

Let $\mathbf{y}^{(k)} = (y_1, y_2, \dots, y_k)$ represent the **running history vector of observed sensor readings** up to the current inspection milestone. Before deploying the sensors, engineers utilize an initial prior distribution $f_{\Theta}^{(0)}(\theta)$ over the domain $(0, 1]$ based on historical manufacturing specifications. As the sensor stream arrives, the posterior distribution calculated at step $k-1$ serves directly as the prior distribution for step $k$.

---

### **Tasks**

#### **1. Prior Belief Boundaries**

Before data collection begins, engineers assume the component is highly likely to be healthy, modeling this using a bounded Beta distribution as the initial prior: $\Theta \sim \text{Beta}(8, 1.5)$.

* Plot this initial prior density function using Plotly over the restricted physical domain $\theta \in [0.01, 1.0]$.
* Calculate the expected prior stiffness efficiency $\mathbb{E}[\Theta^{(0)}]$ analytically. Explain why this specific distribution serves as an appropriate initial prior for an engineering component assumed to be healthy.

#### **2. Structural Likelihood Formulation**

Using the change of variables or properties of the log-normal distribution, write down the mathematical likelihood contribution $L(y_k \mid \theta)$ of a *single* continuous sensor measurement $y_k$ at inspection step $k$, given the true stiffness factor $\theta$. Following this, write down the joint likelihood function for the running history vector $\mathbf{y}^{(k)}$.

#### **3. Mathematical Formulation of the Non-Conjugate Grid Update**

Explain why an exact closed-form analytical solution for the posterior density $f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)})$ does not exist when combining a Beta prior with this log-normal structural likelihood. Write down the recursive relationship for the posterior density at step $k$ up to a proportionality constant.

#### **4. Running Point Estimates**

Because a closed-form formula is unavailable, we must define point estimators through numerical integration. Write down the definite integral equations over the bounded domain $(0, 1]$ required to compute:

* The **Running Posterior Mean** ($\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$)
* The **Running Maximum A Posteriori** ($\widehat{\theta}_{\mathrm{MAP}}^{(k)}$)

#### **5. Algorithmic Grid Approximation and Normalization**

Describe the step-by-step numerical procedure to maintain this distribution on a discrete grid of $\theta$-values. Explicitly state how you would handle the boundary limits computationally and how you would perform the sequential normalization step using the trapezoidal rule after a new sensor reading $y_k$ is observed.

#### **6. Performance Tracking and Degradation Convergence Analysis**

Suppose an impact occurs, and the true, hidden remaining stiffness drops to $\theta_{\text{true}} = 0.68$. Write a Python script using Plotly to simulate an engineered monitoring timeline across $n = 15$ continuous sensor measurements ($K_{\text{nominal}} = 50.0 \text{ kN/mm}$, $\sigma = 0.15$):

* **Simulate Sensor Stream:** Programmatically generate noisy sensor readings $y_k$ by drawing random values from the underlying log-normal physics model centered at $\theta_{\text{true}}$.
* **Track Estimators:** Loop sequentially through each step. At each step, update the unnormalized grid, normalize it via `np.trapezoid`, and compute both $\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$ and $\widehat{\theta}_{\mathrm{MAP}}^{(k)}$.
* **Visualize Curves & Timeline:** Generate two plots:
1. A plot showing the progression of the full posterior density curves at milestones $k \in \{0, 1, 2, 5, 10, 15\}$.
2. A line chart tracking the convergence of both $\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$ and $\widehat{\theta}_{\mathrm{MAP}}^{(k)}$ from step $0$ to $15$ against a horizontal reference line at $\theta_{\text{true}} = 0.68$.


* **Analysis:** Evaluate the behavior of the distribution. How many sensor readings did it take for the system to overcome the initially optimistic "healthy" prior and confidently isolate the 68% damage state? What does the narrowing of the density curves imply about structural safety thresholds?

### 1. Prior Belief Boundaries

Before data collection begins, the initial prior is $\Theta \sim \text{Beta}(8, 1.5)$.

The expected value (mean) of a Beta distribution analytically is:
$$
\mathbb{E}[\Theta^{(0)}] = \frac{\alpha}{\alpha + \beta} = \frac{8}{8 + 1.5} = \frac{8}{9.5} \approx 0.842
$$

**Why this is appropriate:**
For an engineering component presumed to be healthy before deployment, we expect the remaining stiffness factor $\theta$ to be close to $1.0$. The $\text{Beta}(8, 1.5)$ distribution perfectly encodes this engineering intuition: its density is heavily left-skewed (mass shifted to the right), peaking near $1.0$ and tapering off toward $0$. It strictly bounds the parameter to the physically allowable $(0, 1]$ interval, naturally preventing impossible negative stiffness states.

---



### 2. Structural Likelihood Formulation

The measurement model is $y_k = \theta \cdot K_{\text{nominal}} \cdot e^{\epsilon_k}$ with $\epsilon_k \sim \mathscr{N}(0, \sigma^2)$.
By taking the natural logarithm of both sides, we get:
$$
\ln(y_k) = \ln(\theta \cdot K_{\text{nominal}}) + \epsilon_k
$$
This indicates that $\ln(y_k)$ is normally distributed with mean $\ln(\theta K_{\text{nominal}})$ and variance $\sigma^2$. Therefore, $y_k$ follows a log-normal distribution.

The likelihood of a **single sensor measurement** $y_k$ given $\theta$ is:
$$
L(y_k \mid \theta) = \frac{1}{y_k \sigma \sqrt{2\pi}} \exp \left( - \frac{(\ln(y_k) - \ln(\theta K_{\text{nominal}}))^2}{2\sigma^2} \right)
$$

Assuming each sensor reading is conditionally independent given $\theta$, the **joint likelihood function** for the running history vector $\mathbf{y}^{(k)}$ is the product of the individual likelihoods:
$$
L(\mathbf{y}^{(k)} \mid \theta) = \prod_{i=1}^k \frac{1}{y_i \sigma \sqrt{2\pi}} \exp \left( - \frac{(\ln(y_i) - \ln(\theta K_{\text{nominal}}))^2}{2\sigma^2} \right)
$$

### 3. Mathematical Formulation of the Non-Conjugate Grid Update

**Why it is Non-Conjugate:**
Conjugacy requires the prior and the likelihood to share a mathematical structure such that their product yields a known, closed-form posterior. Multiplying a polynomial Beta prior ($\theta^{\alpha-1}(1-\theta)^{\beta-1}$) by a logarithmic exponential term (the log-normal likelihood) creates a complex transcendental function. There is no standard probability distribution that matches this resulting shape, meaning an exact algebraic update for the parameters does not exist.

**Recursive Relationship:**
We must update the distribution sequentially using Bayes' theorem up to a proportionality constant:
$$
f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \propto L(y_k \mid \theta) f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)})
$$

---



### 4. Running Point Estimates

Since we lack a closed-form distribution, we evaluate our point estimates using the normalized numerical density. The normalization constant is the integral of the unnormalized posterior over the domain $(0, 1]$.

*   **Running Posterior Mean** ($\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$):
    $$
    \widehat{\theta}_{\mathrm{Bayes}}^{(k)} = \int_{0}^{1} \theta \cdot f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \, d\theta
    $$

*   **Running Maximum A Posteriori** ($\widehat{\theta}_{\mathrm{MAP}}^{(k)}$):
    The MAP is the mode of the distribution. It requires finding the $\theta$ that maximizes the posterior, evaluated by defining the normalized density via integration:
    $$
    \widehat{\theta}_{\mathrm{MAP}}^{(k)} = \underset{\theta \in (0,1]}{\operatorname{arg\,max}} \left( \frac{L(y_k \mid \theta) f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)})}{\int_{0}^{1} L(y_k \mid s) f_{\Theta \mid \mathbf{Y}^{(k-1)}}(s \mid \mathbf{y}^{(k-1)}) \, ds} \right)
    $$

### 5. Algorithmic Grid Approximation and Normalization

To implement this on a computer without analytical integrals:

1.  **Grid Initialization & Boundary Handling:** Create a dense linear array of $\theta$ values. To handle the boundary limit computationally and prevent a `math domain error` or `division by zero` in the $\ln(\theta)$ likelihood term, we start the grid slightly above zero (e.g., $\theta \in [0.01, 1.0]$).
2.  **Evaluate Prior:** Calculate the Beta PDF across this grid to form the initial state array.
3.  **Sequential Update:** For each incoming $y_k$:
    *   Evaluate the log-normal likelihood function across the entire $\theta$ grid using $y_k$.
    *   Multiply the likelihood array element-wise by the previous step's posterior array.
4.  **Trapezoidal Normalization:** Compute the area under the new unnormalized curve using `np.trapezoid(unnormalized_array, theta_grid)`. Divide the `unnormalized_array` by this scalar area to guarantee it integrates to $1$, yielding a valid PDF for the next step.

---



### 6. Performance Tracking and Degradation Convergence Analysis

The following cell simulates the engineered monitoring timeline across $n = 15$ continuous sensor measurements and visualizes the density curves and estimator convergence.

In [1]:
import numpy as np
import plotly.graph_objects as go
from scipy.stats import beta

# Parameters
np.random.seed(42)
n_steps = 15
K_nom = 50.0
sigma = 0.15
theta_true = 0.68

# Grid setup (avoiding 0 for log-normal stability)
theta_grid = np.linspace(0.01, 1.0, 1000)

# Initialize prior
posterior = beta.pdf(theta_grid, 8, 1.5)
posterior /= np.trapezoid(posterior, theta_grid)  # Ensure exact normalization

# Storage for tracking
bayes_estimates = [np.trapezoid(theta_grid * posterior, theta_grid)]
map_estimates = [theta_grid[np.argmax(posterior)]]
saved_posteriors = {0: np.copy(posterior)}
milestones = [1, 2, 5, 10, 15]

# Simulate True Sensor Stream
# y_k = theta_true * K_nom * exp(epsilon)
epsilon = np.random.normal(0, sigma, n_steps)
sensor_stream = theta_true * K_nom * np.exp(epsilon)

# Sequential Bayesian Update
for k, y_k in enumerate(sensor_stream, start=1):
    # Calculate likelihood on the grid
    # L(y | theta) for log-normal
    likelihood = (1.0 / (y_k * sigma * np.sqrt(2 * np.pi))) * \
                 np.exp(-((np.log(y_k) - np.log(theta_grid * K_nom))**2) / (2 * sigma**2))

    # Update unnormalized posterior
    unnormalized = likelihood * posterior

    # Normalize using Trapezoidal rule
    area = np.trapezoid(unnormalized, theta_grid)
    posterior = unnormalized / area

    # Save milestones
    if k in milestones:
        saved_posteriors[k] = np.copy(posterior)

    # Calculate estimators
    bayes_est = np.trapezoid(theta_grid * posterior, theta_grid)
    map_est = theta_grid[np.argmax(posterior)]

    bayes_estimates.append(bayes_est)
    map_estimates.append(map_est)

In [2]:
# --- Plot 1: Posterior Density Curves ---
fig1 = go.Figure()
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']
for i, (k, post) in enumerate(saved_posteriors.items()):
    fig1.add_trace(go.Scatter(x=theta_grid, y=post, mode='lines',
                              name=f'Step {k} (Prior)' if k==0 else f'Step {k}',
                              line=dict(color=colors[i])))
fig1.add_vline(x=theta_true, line_dash="dash", line_color="black", annotation_text="True θ = 0.68")
fig1.update_layout(title="Progression of Structural Health Posterior Density",
                   xaxis_title="Stiffness Efficiency (θ)", yaxis_title="Density",
                   template="plotly_white")
fig1.show()

In [3]:
# --- Plot 2: Estimator Convergence ---
fig2 = go.Figure()
steps = np.arange(0, n_steps + 1)
fig2.add_trace(go.Scatter(x=steps, y=bayes_estimates, mode='lines+markers', name='Bayes Estimate (Mean)'))
fig2.add_trace(go.Scatter(x=steps, y=map_estimates, mode='lines+markers', name='MAP Estimate (Mode)'))
fig2.add_hline(y=theta_true, line_dash="dash", line_color="black", annotation_text="True θ = 0.68")
fig2.update_layout(title="Convergence of Point Estimators to True Damage State",
                   xaxis_title="Inspection Step (k)", yaxis_title="Estimated Stiffness (θ)",
                   template="plotly_white", xaxis=dict(dtick=1))
fig2.show()

### Analysis of the System's Behavior

*   **Overcoming the Prior:** The initial Beta prior strongly biased the system toward a healthy state ($\approx 0.84$). However, looking at the estimator convergence plot, it takes approximately **3 to 5 sensor readings** for the incoming likelihood data to overpower the optimistic prior. By step 5, both the Bayes and MAP estimates drop sharply and lock into the vicinity of the true damage state ($0.68$).
*   **Implications of Curve Narrowing:** As $k$ increases to 10 and 15, the density curves transition from a broad spread to a highly localized, sharp peak around $0.68$. In engineering terms, this narrowing variance represents a vast reduction in measurement uncertainty. Tighter density curves allow automated monitoring systems to confidently trigger maintenance alerts without false positives, optimizing structural safety thresholds while preventing unnecessary grounding or downtime.